# Análise exploratória dos dados de câncer no Brasil

Dados provenientes do INCA (Instituto Nacional de Câncer) foram carregados para análise. Fonte dos dados: [INCA - Registro Hospitalar de Câncer](https://irhc.inca.gov.br/RHCNet/visualizaTabNetExterno.action) e [Kaggle](https://www.kaggle.com/dfsets/rafatrindade/onco-360?select=raw_inca_registro_hospitalar.parquet).

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from pyspark.sql import functions as F

In [0]:
df = spark.read.parquet('/Volumes/workspace/default/cancer_data/raw_inca_registro_hospitalar.parquet')
display(df.limit(10))

In [0]:
mapeamento_colunas = {
    "ALCOOLIS": "historico_alcoolismo",
    "ANOPRIDI": "data_diagnostico",
    "ANTRI": "ano_triagem",
    "BASMAIMP": "base_diagnostico_mais_importante",
    "BASDIAGSP": "base_diagnostico_microscopica",
    "CLIATEN": "clinica_primeiro_atendimento",
    "CLITRAT": "clinica_inicio_tratamento",
    "CNES": "codigo_cnes_hospital",
    "DATAINITRT": "data_inicio_tratamento_hospital",
    "DATAOBITO": "data_obito",
    "DATAPRICON": "data_primeira_consulta",
    "DIAGANT": "diagnostico_tratamento_anteriores",
    "DTDIAGNO": "data_primeiro_diagnostico",
    "DTINITRT": "ano_inicio_tratamento_hospital",
    "DTPRICON": "ano_primeira_consulta",
    "DTTRIAGE": "data_triagem",
    "ESTADIAG": "estadiamento_clinico_grupo",
    "ESTADIAM": "estadiamento_clinico_tnm",
    "ESTADRES": "uf_procedencia",
    "ESTCONJ": "estado_conjugal",
    "ESTDFIMT": "estado_doenca_fim_tratamento",
    "EXDIAG": "exames_relevantes_diagnostico",
    "HISTFAMC": "historico_familiar_cancer",
    "IDADE": "idade",
    "INSTRUC": "escolaridade",
    "LATERALI": "lateralidade_tumor",
    "LOCALNAS": "local_nascimento",
    "LOCTUDET": "localizacao_primaria_3_digitos",
    "LOCTUPRI": "localizacao_primaria_detalhada",
    "LOCTUPRO": "localizacao_provavel_primaria",
    "MAISUMTU": "mais_de_um_tumor_primario",
    "MUUH": "municipio_hospital",
    "OCUPACAO": "ocupacao_principal",
    "ORIENC": "origem_encaminhamento",
    "OUTROESTA": "outros_estadiamentos_clinicos",
    "PRITRATH": "primeiro_tratamento_hospital",
    "PROCEDEN": "codigo_municipio_residencia",
    "PTNM": "ptnm",
    "RACACOR": "raca_cor",
    "RZNTR": "razao_nao_tratamento_hospital",
    "SEXO": "sexo",
    "TABAGISM": "historico_tabagismo",
    "TIPOHIST": "tipo_histologico_tumor",
    "TNM": "tnm",
    "TPCASO": "tipo_caso",
    "UFUH": "uf_hospital",
    "VALOR_TOT": "text"
}

for nome_original, novo_nome in mapeamento_colunas.items():
    if nome_original in df.columns:
        df = df.withColumnRenamed(nome_original, novo_nome)

df.printSchema()

#### Colunas deletadas

- text: todos tem o mesmo valor
- data_triagem: irrelevante em comparação com outras datas e tem muitos valores nan
- municipio_hospital: decidimos deixar apenas a UF
- codigo_cnes_hospital: como dados do hospital deixamos apenas a UF
- ano_inicio_tratamento_hospital: ja tem outra feature semelhante
- localizacao_provavel_primaria: 99% de valores nulos
- lateralidade_tumor: como n temos a localização do tumor (loctupro) acreditamos que essa feature n agrega valor
- ano_primeira_consulta: ja tem uma feature com a data completa, essa é só o ano e além disso tem mais valores nan
- ano_triagem: ja tem uma feature com a data completa, essa é só o ano e além disso tem mais valores nan
- estado_conjugal: irrelevante para o modelo
- codigo_municipio_residencia: acreditamos que a localização do hospital já cobre esse aspecto
- clinica_inicio_tratamento: irrelevante, pouco contato com o paciente, hospital tende a ser mais importante
- clinica_primeiro_atendimento: irrelevante, pouco contato com o paciente, hospital tende a ser mais importante
- local_nascimento: acreditamos ser mais importante a localização do hospital do que a do nascimento da pessoa, visto que ela pode ter se mudado e nem ter mais relação com a cidade natal
- estadiamento_clinico_grupo: muitos valores nan
- estadiamento_clinico_tnm: muitos valores nan
- data_diagnostico: tem data_primeiro_diagnostico que é mais completa
- outros_estadiamentos_clinicos: falta de informação
- localizacao_primaria_3_digitos: localizacao_primaria_detalhada tem a mesma informação e de forma mais completa

In [0]:
colunas_para_remover = ['text', 'data_triagem', 'municipio_hospital', 'codigo_cnes_hospital', 'ano_inicio_tratamento_hospital', 'localizacao_provavel_primaria', 'lateralidade_tumor', 'ano_primeira_consulta', 'ano_triagem', 'estado_conjugal', 'codigo_municipio_residencia', 'clinica_inicio_tratamento', 'clinica_primeiro_atendimento', 'local_nascimento', 'estadiamento_clinico_grupo', 'estadiamento_clinico_tnm', 'data_diagnostico', 'outros_estadiamentos_clinicos', 'localizacao_primaria_3_digitos']

df = df.drop(*colunas_para_remover)

In [0]:
df = df.replace('nan', None)

In [0]:
# 1. Calculando nulos para todas as colunas
# Usamos .isNull() para nulos reais e comparamos com 'nan' (string) 
null_counts = df.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c) == 'nan') | (F.col(c) == ''), c)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

# 2. Convertendo para Pandas para o Gráfico
df_missing = pd.DataFrame(list(null_counts.items()), columns=['Coluna', 'Total_Nulos'])

# 3. Filtrando apenas as que possuem nulos e ordenando
df_missing = df_missing[df_missing['Total_Nulos'] > 0].sort_values(by='Total_Nulos', ascending=False)

# 4. Plotando
if not df_missing.empty:
    plt.figure(figsize=(15, 8))
    sns.barplot(data=df_missing, x='Coluna', y='Total_Nulos', palette='viridis')
    
    plt.xticks(rotation=85) # Rotação maior pois você tem muitos nomes de colunas
    plt.title('Valores Faltantes por Coluna (Dataset Câncer)', fontsize=16)
    plt.ylabel('Quantidade de Nulos')
    plt.xlabel('Nome da Coluna')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum valor nulo ou 'nan' encontrado!")

In [0]:
print(df.count())
df = df.dropna(subset=[col for col in df.columns if col != 'data_obito'])
print(df.count())

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "True").saveAsTable("after_nan_table")

## Coerência dos dados

In [0]:
df = spark.table("after_nan_table")

In [0]:
date_cols = [
    'data_primeiro_diagnostico', 
    'data_inicio_tratamento_hospital', 
    'data_obito', 
    'data_primeira_consulta'
]

df = df.withColumns({
    c: F.expr(f"try_to_date({c}, 'dd/MM/yyyy')") 
    for c in date_cols if c in df.columns
})

display(df.limit(3))

In [0]:
# Criamos uma lista de expressões para processar tudo em uma única passagem pelos dados
agg_exprs = []
for c in date_cols:
    if c in df.columns:
        agg_exprs.extend([
            F.min(c).alias(f"{c}_min"),
            F.max(c).alias(f"{c}_max"),
            F.count(F.when(F.col(c).isNull(), c)).alias(f"{c}_nan")
        ])

# Executa a agregação e traz o resultado
stats = df.select(agg_exprs).collect()[0]

for c in date_cols:
    if c in df.columns:
        print(f"\n{c}:")
        print(f"  Min: {stats[f'{c}_min']}")
        print(f"  Max: {stats[f'{c}_max']}")
        print(f"  NaN: {stats[f'{c}_nan']}")

In [0]:
min_date, max_date = '1900-01-01', '2023-12-31'

# 1. Condições de Range de Data
range_cond = (
    (F.col('data_primeiro_diagnostico').between(min_date, max_date)) &
    (F.col('data_inicio_tratamento_hospital').between(min_date, max_date) | F.col('data_inicio_tratamento_hospital').isNull()) &
    (F.col('data_obito').between(min_date, max_date) | F.col('data_obito').isNull())
)

# 2. Condições de Inconsistência Lógica (Invertidas para manter o que é válido)
# Regra: Diagnóstico <= Tratamento <= Óbito
logic_cond = (
    ((F.col('data_inicio_tratamento_hospital') >= F.col('data_primeiro_diagnostico')) | F.col('data_inicio_tratamento_hospital').isNull()) &
    ((F.col('data_obito') >= F.col('data_primeiro_diagnostico')) | F.col('data_obito').isNull()) &
    ((F.col('data_obito') >= F.col('data_inicio_tratamento_hospital')) | F.col('data_obito').isNull() | F.col('data_inicio_tratamento_hospital').isNull())
)

# Aplica tudo em um único scan
df = df.filter(range_cond & logic_cond)

In [0]:
# Excluindo datas null (vindas do errors=coerce), exceto para data_obito em que null pode significar que a pessoa esta viva (ent precisamos dessa info)
#FIXME pq não data_primeira_consulta
df = df.dropna(subset=['data_primeiro_diagnostico', 'data_inicio_tratamento_hospital'])
display(df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]))

In [0]:
df = df.withColumn('idade', F.expr("try_cast(idade as int)"))

df = df.filter(
    (F.col('idade').isNotNull()) & 
    (F.col('idade') >= 0) & 
    (F.col('idade') <= 100)
)

In [0]:
# FIXME demora bastante, é mais demonstrativo, podemos apagar depois, não precisa rodar

# 1. Filtramos os não nulos para facilitar
df = df.filter(F.col('tnm').isNotNull())
total_nao_nulos = df.count()

# 2. Calculamos todas as métricas de uma vez
stats = df.agg(
    F.count(F.when(F.col('tnm').rlike('[A-Za-z]'), 1)).alias('com_letras'),
    F.count(F.when(F.col('tnm') == '999', 1)).alias('valor_999'),
    F.count(F.when(F.col('tnm') == '888', 1)).alias('valor_888'),
    F.count(F.when(F.length(F.col('tnm')) < 3, 1)).alias('menos_3'),
    F.count(F.when(F.length(F.col('tnm')) > 3, 1)).alias('mais_3')
).collect()[0]

print(f"Total tnm não nulos: {total_nao_nulos}")
print(f"Com letras: {stats['com_letras']}")
print(f"Com valor '999': {stats['valor_999']}")
print(f"Com valor '888': {stats['valor_888']}")
print(f"Com menos de 3 caracteres: {stats['menos_3']}")
print(f"Com mais de 3 caracteres: {stats['mais_3']}")

In [0]:
# 1. Tratamento e Separação
df = (
    df.withColumn("tnm_temp", F.translate(F.col("tnm"), "X", "-"))
    .withColumn("t_tnm", F.expr("try_cast(substring(tnm_temp, 1, 1) as int)"))
    .withColumn("n_tnm", F.expr("try_cast(substring(tnm_temp, 2, 1) as int)"))
    .withColumn("m_tnm", F.expr("try_cast(substring(tnm_temp, 3, 1) as int)"))
    .drop("tnm_temp")
)

# FIXME uncomment!
# 2. Filtragem (Mantém -1 a 4 para T, -1 a 3 para N e -1 a 1 para M)
# df = df.filter(
#     (F.col("T_tnm").between(-1, 4)) 
#     & (F.col("N_tnm").between(-1, 3)) 
#     & (F.col("M_tnm").between(-1, 1)) 
# )

In [0]:
df.count()

In [0]:
# Excluindo linhas que contenham alguma feature com valor fora do dominio valido (https://raw.githubusercontent.com/rafa-trindade/oncoped-360/main/docs/dicionario_inca_registro_hospitalar.pdf)
validos = {
    "tipo_caso": {"1", "2"},
    "sexo": {"1", "2"},
    "raca_cor": {"1", "2", "3", "4", "5"},
    "escolaridade": {"1", "2", "3", "4", "5", "6"},
    "historico_familiar_cancer": {"1", "2"},
    "historico_alcoolismo": {"1", "2", "3", "4", "8"},
    "historico_tabagismo": {"1", "2", "3", "4", "8"},
    "origem_encaminhamento": {"1", "2", "3", "8"},
    "exames_relevantes_diagnostico": {"1", "2", "3", "4", "5", "8"},
    "diagnostico_tratamento_anteriores": {"1", "2", "3", "4"},
    "base_diagnostico_mais_importante": {"1", "2", "3", "4", "5", "6", "7"},
    "mais_de_um_tumor_primario": {"1", "2", "3"},
    "razao_nao_tratamento_hospital": {"1", "2", "3", "4", "5", "6", "7", "8"},
    "primeiro_tratamento_hospital": {"1", "2", "3", "4", "5", "6", "7", "8"},
    "estado_doenca_fim_tratamento": {"1", "2", "3", "4", "5", "6", "8"},
    "base_diagnostico_microscopica": {"1", "2", "3"},
}

ufs_validas = {
    "AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS",
    "MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC",
    "SP","SE","TO"
}

# 1. Filtros de Categorias Fixas
for col, valores in validos.items():
    if col in df.columns:
        df = df.filter(F.col(col).isin(list(valores)))

# 2. Filtros de UFs
for col in ["uf_procedencia", "uf_hospital"]:
    if col in df.columns:
        df = df.filter(F.col(col).isin(list(ufs_validas)))

# 3. Filtros com Regex (rlike)
if "ocupacao_principal" in df.columns:
    # O regex [1-9]\d* garante que comece com 1-9 (evita 0 ou vazios)
    df = df.filter(F.col("ocupacao_principal").rlike("^[1-9]\d*$"))

if "tipo_histologico_tumor" in df.columns:
    df = df.filter(F.col("tipo_histologico_tumor").rlike("^(8|9)\d{3}/[012369]$"))

if "localizacao_primaria_detalhada" in df.columns:
    df = df.filter(F.col("localizacao_primaria_detalhada").rlike("^C(0[0-9]|[1-7][0-9]|80)\.[0-9]$"))

In [0]:
df.count()
display(df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]))

In [0]:
comparativo_nulos = df.groupBy(F.year('data_primeiro_diagnostico').alias('ano')) \
    .agg(
        F.count(F.when(F.col('t_tnm').isNull(), 1)).alias('t_tnm_nulos'),
        F.count(F.when(F.col('n_tnm').isNull(), 1)).alias('n_tnm_nulos'), 
        F.count(F.when(F.col('m_tnm').isNull(), 1)).alias('m_tnm_nulos'),
    ) \
    .orderBy('ano') \
    .toPandas()

df_plot = comparativo_nulos.melt(id_vars='ano', var_name='Tipo', value_name='Total_Nulos')

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_plot, x='ano', y='Total_Nulos', hue='Tipo', marker='o')

plt.title('Comparação de Nulos: TNM vs PTNM ao Longo do Tempo', fontsize=14)
plt.xlabel('Ano de Diagnóstico')
plt.ylabel('Quantidade de Nulos (log)')
plt.yscale('log') # Escala logarítmica como nos anteriores
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.legend(title='Indicador')
plt.tight_layout()
plt.show()

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "True").saveAsTable("after_coerencia_dados")

## Criando novas features

- separar idade em faixas etárias?
- separar datas em anos (ou em periodos?), manter meses e dias?

In [0]:
df = spark.table("after_coerencia_dados")

In [0]:
# 1. Criar STATUS_VITAL
# 0 para mortos (estado '6' ou data_obito preenchida), 1 para vivos
df = df.withColumn(
    "STATUS_VITAL",
    F.when(
        (F.col("estado_doenca_fim_tratamento") == "6") | (F.col("data_obito").isNotNull()), 
        0
    ).otherwise(1)
)

# 2. Criar TEMPO_SEGUIMENTO e TEMPO_ATE_TRATAMENTO
# datediff(fim, inicio) retorna a diferença em dias
df = df.withColumn(
    "TEMPO_SEGUIMENTO",
    F.when(
        F.col("STATUS_VITAL") == 1,
        F.datediff(F.lit("2023-12-31"), F.col("data_primeiro_diagnostico"))
    ).otherwise(
        F.datediff(F.col("data_obito"), F.col("data_primeiro_diagnostico"))
    )
).withColumn(
    "TEMPO_ATE_TRATAMENTO",
    F.datediff(F.col("data_inicio_tratamento_hospital"), F.col("data_primeiro_diagnostico"))
)

In [0]:
#FIXME: essa proporção nao esta muito discrepante para um algoritmo de ml?
status_counts = df['STATUS_VITAL'].value_counts(normalize=True)
plt.figure(figsize=(6, 4))
sns.barplot(x=['Óbito' if i == 0 else 'Vivo' for i in status_counts.index], y=status_counts.values, palette='viridis')
plt.title('Gráfico STATUS_VITAL')
plt.ylabel('Proporção')
plt.xlabel('Status Vital')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [0]:
#TODO: Decodificar outras variaveis relevantes, como loctupri?

In [0]:
#TODO: excluir features de datas desnecessarias, se vamos trabalhar com as variaveis de tempo, entao podemos excluir as datas, se n fica redundante

## Encode

Fazer one-hot encoding das variáveis categóricas selecionadas, como tnm, loctupri, sexo, etc

Para um modelo de rsf, os códigos de CID-O não devem ser usados como valores numéricos brutos, pois isso introduz uma ordem artificial sem significado clínico; em vez disso:

- tratar a variável como categórica, preferencialmente usando a localizacao_primaria_detalhada (CID-O, 4 dígitos) por ser mais informativa. Caso o número de observações seja limitado, usar localizacao_primaria_3_digitos (CID-O, 3 dígitos)
- agrupar categorias raras (por exemplo, localizações com baixa frequência) em uma classe “Outros”
- aplicar one-hot encoding 

In [0]:
#TODO : coisas acima para a feature de CID-O

In [0]:
#TODO: fazer encoding de todas outras variaveis necessarias

## Gráficos

In [0]:
map_sexo = {
    '1': 'Masculino',
    '2': 'Feminino'
}

map_tabagismo = {
    '1': 'Nunca',
    '2': 'Ex-consumidor',
    '3': 'Sim',
    '4': 'Não avaliado',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_alcool = map_tabagismo.copy()

map_racacor = {
    '1': 'Branca',
    '2': 'Preta',
    '3': 'Amarela',
    '4': 'Parda',
    '5': 'Indígena',
    '9': 'Sem informação'
}

map_estdfimt = {
    '1': 'Remissão completa',
    '2': 'Remissão parcial',
    '3': 'Doença estável',
    '4': 'Progressão',
    '5': 'Suporte terapêutico',
    '6': 'Óbito',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_pritrath = {
    '1': 'Nenhum',
    '2': 'Cirurgia',
    '3': 'Radioterapia',
    '4': 'Quimioterapia',
    '5': 'Hormonioterapia',
    '6': 'Transplante de medula óssea',
    '7': 'Imunoterapia',
    '8': 'Outros',
    '9': 'Sem informação'
}

map_base_diagnostico_mais_importante = {
    '1': 'Clínica',
    '2': 'Pesquisa clínica',
    '3': 'Exame por imagem',
    '4': 'Marcadores tumorais',
    '5': 'Citologia',
    '6': 'Histologia da metástase',
    '7': 'Histologia do tumor primário',
    '9': 'Sem informação'
}

map_exdiag = {
    '1': 'Exame clínico / Patologia clínica',
    '2': 'Exames por imagem',
    '3': 'Endoscopia / Cirurgia exploradora',
    '4': 'Anatomia patológica',
    '5': 'Marcadores tumorais',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_orienc = {
    '1': 'SUS',
    '2': 'Não SUS',
    '3': 'Conta própria',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_tp_caso = {
    '1': 'Analítico',
    '2': 'Não analítico',
}

map_status_vital = {
    0: 'Óbito',
    1: 'Vivo'
}

In [0]:
df['data_primeiro_diagnostico_Year'] = pd.to_datetime(df['data_primeiro_diagnostico'], errors='coerce').dt.year
obito_counts = df[df['data_obito'].isna()].groupby('data_primeiro_diagnostico_Year').size()
plt.figure(figsize=(10, 6))
sns.barplot(x=obito_counts.index, y=obito_counts.values, palette='viridis')
plt.title("Count of nan in data_obito by Year of data_primeiro_diagnostico")
plt.xlabel("Year of data_primeiro_diagnostico")
plt.ylabel("Count of nan in data_obito")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['sexo'].map(map_sexo).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Distribuição por Sexo')
axes[0].set_ylabel('Pacientes')

df['raca_cor'].map(map_racacor).value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Distribuição por Raça/Cor')

df['idade'].dropna().plot(kind='hist', bins=20, ax=axes[2])
axes[2].set_title('Distribuição de Idade')
axes[2].set_xlabel('Idade')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['historico_tabagismo'].map(map_tabagismo).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Histórico de Tabagismo')

df['historico_alcoolismo'].map(map_alcool).value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Histórico de historico_alcoolismomo')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['base_diagnostico_mais_importante'].map(map_base_diagnostico_mais_importante).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Base Mais Importante do Diagnóstico')

df['exames_relevantes_diagnostico'].astype(str).str.strip().map(map_exdiag).fillna('Sem informação') \
    .value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Exames para Diagnóstico')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['primeiro_tratamento_hospital'].map(map_pritrath).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Primeiro Tratamento Recebido')

df['estado_doenca_fim_tratamento'].map(map_estdfimt).value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Estado da Doença ao Final do Tratamento')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

df.loc[df['STATUS_VITAL'] == 0, 'idade'].dropna().plot(
    kind='hist', bins=20, alpha=0.7, ax=axes[0]
)
axes[0].set_title('Idade - Não Óbito')
axes[0].set_xlabel('Idade')

df.loc[df['STATUS_VITAL'] == 1, 'idade'].dropna().plot(
    kind='hist', bins=20, alpha=0.7, ax=axes[1]
)
axes[1].set_title('Idade - Óbito')
axes[1].set_xlabel('Idade')

plt.tight_layout()
plt.show()

In [0]:
tpcaso = (
    df['tipo_caso']
    .astype(str)
    .str.strip()
    .map(map_tp_caso)
    .fillna('Sem informação')
)

cross = pd.crosstab(tpcaso, df['STATUS_VITAL'].map(map_status_vital))
cross_prop = cross.div(cross.sum(axis=1), axis=0)

cross_prop.plot(
    kind='bar',
    stacked=True,
    figsize=(8, 5)
)

plt.title('Proporção de Óbito por Tipo de Caso')
plt.ylabel('Proporção')
plt.xlabel('Tipo de Caso')
plt.legend(title='Status Vital')
plt.tight_layout()
plt.show()